In [ ]:
import pandas as pd

activities_clean = pd.read_csv(
    "../data/processed/activities_clean.csv"
)

compounds = pd.read_csv(
    "../data/raw/chembl_leishmania_compounds.csv",
    sep=";",
    engine="python",
    on_bad_lines="skip"
)
compounds.columns.tolist()

## Merging Activity and Descriptor Data

The cleaned activity dataset was merged with the compound descriptor dataset using ChEMBL compound identifiers. This step combined biological activity information (pIC50) with molecular descriptors required for machine learning model development.

In [ ]:
merged_df = pd.merge(
    activities_clean,
    compounds,
    left_on="Molecule ChEMBL ID",
    right_on="Compound ChEMBL ID",
    how="inner"
)
print(merged_df.shape)
merged_df.head()

In [ ]:
(
    merged_df["Smiles_x"] ==
    merged_df["Smiles_y"]
).value_counts()

## Verification of SMILES Consistency

SMILES representations from the activity and descriptor datasets were compared following the merge. The vast majority of compounds (5,525 out of 5,533) showed identical molecular structures across both datasets, indicating successful matching of compound records. A small number of discrepancies were retained for further inspection.

In [ ]:
merged_df[
    merged_df["Smiles_x"] != merged_df["Smiles_y"]
][
    [
        "Molecule ChEMBL ID",
        "Smiles_x",
        "Smiles_y"
    ]
]

In [ ]:
merged_df=merged_df.dropna(subset=['Smiles_x'])
print(merged_df.shape)

## Verification of Molecular Structures

SMILES representations from both datasets were compared after merging. All available structures were found to be consistent between datasets. A small number of compounds contained missing SMILES information and were removed prior to model development.

In [ ]:
merged_df.isnull().sum().sort_values(
    ascending=False
)

In [ ]:
descriptor_cols = [
    "Molecular Weight",
    "AlogP",
    "Polar Surface Area",
    "HBA",
    "HBD",
    "#RO5 Violations",
    "#Rotatable Bonds",
    "QED Weighted",
    "Aromatic Rings",
    "Heavy Atoms",
    "Np Likeness Score"
]
merged_df = merged_df.dropna(
    subset=descriptor_cols
)

print(merged_df.shape)

## Handling Missing Descriptor Values

Several molecular descriptor columns contained a small number of missing values. Because these missing records represented a very small proportion of the dataset, compounds lacking descriptor information were removed to ensure a complete feature set for machine learning analysis.

In [ ]:
model_df = merged_df[
    [
        "Molecule ChEMBL ID",
        "Smiles_x",
        "pIC50",
        "Molecular Weight",
        "AlogP",
        "Polar Surface Area",
        "HBA",
        "HBD",
        "#RO5 Violations",
        "#Rotatable Bonds",
        "QED Weighted",
        "Aromatic Rings",
        "Heavy Atoms",
        "Np Likeness Score"
    ]
].copy()

## Selection of Modeling Features

Relevant molecular descriptors and the target activity variable (pIC50) were selected to construct the final modeling dataset. Metadata and non-informative identifiers were excluded, retaining only descriptors expected to contribute to activity prediction.

In [ ]:
model_df.rename(
    columns={
        "Smiles_x": "SMILES",
        "Molecular Weight": "Molecular_Weight",
        "Polar Surface Area": "TPSA",
        "#RO5 Violations": "RO5_Violations",
        "#Rotatable Bonds": "Rotatable_Bonds",
        "QED Weighted": "QED",
        "Aromatic Rings": "Aromatic_Rings",
        "Heavy Atoms": "Heavy_Atoms",
        "Np Likeness Score": "NP_Likeness"
    },
    inplace=True
)

model_df.head()

## Standardization of Feature Names

Descriptor column names were standardized to improve readability and facilitate downstream analysis and model development.

In [ ]:
model_df.to_csv(
    "../data/processed/leishmania_ml_dataset.csv",
    index=False
)

In [ ]:
model_df.head()